# Day 12: Deployment — Streamlit + FastAPI 🚀

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Streamlit chatbot — multi-turn UI wrapping your RAG + Memory agent
2. FastAPI endpoint — expose the agent as an HTTP API
3. Streaming in Streamlit — word-by-word response (ChatGPT effect)
4. Connect Streamlit → FastAPI — full-stack local application
5. API key authentication — simple production security

**Note:** Streamlit and FastAPI apps run as separate processes — not as notebook cells. Each section writes a `.py` file to disk. You then run it from your terminal.

**Time:** ~2 hours | **Prerequisites:** Days 10-11 complete

---

## 0. Setup — Agent Module

First, we write the shared agent code to a reusable Python file. Both Streamlit and FastAPI will import from this file.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")

In [ ]:
# Write agent.py — the shared agent module used by both Streamlit and FastAPI
agent_code = '''
"""
agent.py — Shared RAG + Memory Agent
Agentic AI Hands-On Course | Dr. Kanthi Kiran Sirra
Import this from Streamlit or FastAPI — do not run directly.
"""
import os
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# ── LLM and embedding model ───────────────────────────────
llm     = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ── Knowledge base (in-memory ChromaDB) ──────────────────
client = chromadb.Client()
try:
    client.delete_collection("agentic_ai_kb")
except:
    pass
collection = client.create_collection("agentic_ai_kb")

DOCUMENTS = [
    {"id":"doc_001", "topic":"LangGraph Overview",
     "text":"LangGraph is an open-source library by LangChain Inc. for stateful multi-actor LLM applications. It models workflows as directed graphs with nodes (steps) and edges (transitions). State is a shared TypedDict passed between nodes. MemorySaver enables persistent state via thread IDs."},
    {"id":"doc_002", "topic":"RAG Architecture",
     "text":"RAG (Retrieval-Augmented Generation) enhances LLM responses by retrieving relevant documents first. The pipeline: load documents, chunk into 500-1000 char pieces, embed with all-MiniLM-L6-v2, store in ChromaDB. At query time, embed the question, retrieve top-k chunks by cosine similarity, pass chunks as context to the LLM."},
    {"id":"doc_003", "topic":"Agent Memory Systems",
     "text":"Agents need memory for multi-turn conversations. Short-term memory is a list of dicts with role (user/assistant) and content. A sliding window of 6 messages (3 turns) keeps token costs bounded. LangGraph MemorySaver saves full state across invoke() calls using thread IDs."},
    {"id":"doc_004", "topic":"Self-Reflection Pattern",
     "text":"Self-reflection: LLM generates an answer, then critiques it, then improves it. Works because generating and evaluating are different modes. LangGraph implements it as generate_node -> critique_node -> improve_node with a conditional edge that loops back or exits. Always use a max iterations limit of 3."},
    {"id":"doc_005", "topic":"Groq API",
     "text":"Groq provides fast LLM inference via custom LPU hardware. Free tier: ~100K tokens/day, 30 req/min. llama-3.3-70b-versatile runs at 300+ tokens/sec. Groq API is OpenAI-compatible. ChatGroq in LangChain reads GROQ_API_KEY from environment automatically."},
]

texts = [d["text"] for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()
collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[d["id"] for d in DOCUMENTS],
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

# ── LangGraph state and nodes ─────────────────────────────
class AgentState(TypedDict):
    question:  str
    messages:  List[dict]
    retrieved: str
    answer:    str


def memory_node(state):
    msgs = state.get("messages", []) + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:
        msgs = msgs[-6:]
    return {"messages": msgs}


def retrieval_node(state):
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    context = "\\n\\n".join(results["documents"][0])
    return {"retrieved": context}


def answer_node(state):
    lc_msgs = [SystemMessage(content=f"""Answer using ONLY the context below.
If not in context, say: I don\'t have that information.

CONTEXT:
{state[\'retrieved\']}""")]
    for msg in state.get("messages", [])[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=state["question"]))
    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


def save_node(state):
    msgs = state.get("messages", []) + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": msgs}


# ── Build and compile the graph ───────────────────────────
graph = StateGraph(AgentState)
graph.add_node("memory",   memory_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("answer",   answer_node)
graph.add_node("save",     save_node)

graph.set_entry_point("memory")
graph.add_edge("memory",   "retrieve")
graph.add_edge("retrieve", "answer")
graph.add_edge("answer",   "save")
graph.add_edge("save",     END)

checkpointer = MemorySaver()
agent_app    = graph.compile(checkpointer=checkpointer)


def ask(question: str, thread_id: str = "default") -> str:
    """Ask the agent a question and get an answer."""
    config = {"configurable": {"thread_id": thread_id}}
    result = agent_app.invoke({"question": question}, config=config)
    return result["answer"]


if __name__ == "__main__":
    print("Agent module loaded.")
    print(f"Knowledge base: {collection.count()} documents")
    print(ask("What is RAG?", thread_id="test"))
'''

with open("agent.py", "w", encoding="utf-8") as f:
    f.write(agent_code)

print("✅ agent.py written")
print("   Both Streamlit and FastAPI will import from this file.")

print("✅ agent.py written successfully — ready for Streamlit and FastAPI.")

---
## 1. 🌐 Streamlit Chatbot

Write `streamlit_app.py` then run it. Streamlit opens a browser window at `localhost:8501` automatically.

**Key concepts used:**
- `st.chat_input()` — the input box at the bottom
- `st.chat_message()` — renders user/assistant bubbles
- `st.session_state` — persists messages and thread_id across reruns

In [ ]:
streamlit_app = '''
"""
streamlit_app.py — RAG Chatbot UI
Run with: streamlit run streamlit_app.py
Opens browser at: http://localhost:8501
"""
import streamlit as st
import uuid
from agent import ask

# ── Page config ───────────────────────────────────────────
st.set_page_config(
    page_title="Agentic AI Assistant",
    page_icon="🤖",
    layout="centered"
)

st.title("🤖 Agentic AI Course Assistant")
st.caption("Powered by LangGraph + RAG | Dr. Kanthi Kiran Sirra")

# ── Initialise session state on first run ─────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []

if "thread_id" not in st.session_state:
    # Unique ID per browser session
    st.session_state.thread_id = str(uuid.uuid4())[:8]

# ── Sidebar — show session info + clear button ────────────
with st.sidebar:
    st.header("Session Info")
    st.write(f"Thread ID: `{st.session_state.thread_id}`")
    st.write(f"Messages:  {len(st.session_state.messages)}")

    if st.button("🗑️ Clear conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

    st.divider()
    st.markdown("**Knowledge base topics:**")
    topics = ["LangGraph Overview", "RAG Architecture",
              "Agent Memory Systems", "Self-Reflection Pattern", "Groq API"]
    for t in topics:
        st.write(f"• {t}")

# ── Display conversation history ──────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ── Chat input ────────────────────────────────────────────
if prompt := st.chat_input("Ask about LangGraph, RAG, memory systems..."):

    # Show user message immediately
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    # Get agent answer
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            answer = ask(prompt, thread_id=st.session_state.thread_id)
        st.write(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open("streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_app)

print("✅ streamlit_app.py written")
print()
print("To run:")
print("  1. Open a NEW terminal (separate from this notebook)")
print("  2. cd to this notebook's directory")
print("  3. Activate venv: venv\\Scripts\\activate")
print("  4. Run: streamlit run streamlit_app.py")
print("  5. Browser opens at http://localhost:8501")

### What to test in the Streamlit app:
1. Ask: `"What is LangGraph?"` — should retrieve from knowledge base
2. Ask: `"What was my first question?"` — should remember from session_state
3. Click **Clear conversation** — messages should disappear, new thread_id assigned
4. Ask: `"What is the price of a GPU?"` — should admit it doesn't know (not in KB)

---
## 2. ⚡ FastAPI Endpoint

Write `fastapi_app.py` and run it with uvicorn. The API will be live at `localhost:8000`. FastAPI auto-generates interactive docs at `localhost:8000/docs`.

In [ ]:
fastapi_app = '''
"""
fastapi_app.py — REST API for the RAG Agent
Run with: uvicorn fastapi_app:app --reload --port 8000
Docs at:  http://localhost:8000/docs
"""
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional
import time
from agent import ask, collection

# ── App setup ─────────────────────────────────────────────
app = FastAPI(
    title="Agentic AI Course API",
    description="RAG + Memory agent API — Day 12",
    version="1.0.0"
)

# Allow Streamlit frontend to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:8501"],  # Streamlit
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Request / Response models ─────────────────────────────
class AskRequest(BaseModel):
    question:  str          = Field(..., description="The question to ask", min_length=1)
    thread_id: str          = Field("default", description="Conversation thread ID")

class AskResponse(BaseModel):
    answer:    str
    thread_id: str
    latency_ms: float

class HealthResponse(BaseModel):
    status:   str
    kb_docs:  int
    model:    str

# ── Endpoints ─────────────────────────────────────────────
@app.get("/health", response_model=HealthResponse)
def health():
    """Check if the API is running and the knowledge base is loaded."""
    return HealthResponse(
        status="ok",
        kb_docs=collection.count(),
        model="llama-3.3-70b-versatile"
    )


@app.post("/ask", response_model=AskResponse)
def ask_question(request: AskRequest):
    """Ask the agent a question. Supports multi-turn conversations via thread_id."""
    if not request.question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be empty")

    start = time.time()
    try:
        answer = ask(request.question, thread_id=request.thread_id)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Agent error: {str(e)[:200]}")

    latency = (time.time() - start) * 1000

    return AskResponse(
        answer=answer,
        thread_id=request.thread_id,
        latency_ms=round(latency, 1)
    )


@app.get("/")
def root():
    return {"message": "Agentic AI API running. Visit /docs for interactive documentation."}
'''

with open("fastapi_app.py", "w", encoding="utf-8") as f:
    f.write(fastapi_app)

print("✅ fastapi_app.py written")
print()
print("To run:")
print("  1. Open a NEW terminal")
print("  2. cd to this notebook's directory")
print("  3. Activate venv: venv\\Scripts\\activate")
print("  4. Run: uvicorn fastapi_app:app --reload --port 8000")
print("  5. Open http://localhost:8000/docs for interactive API docs")
print()
print("Test with curl (from a third terminal):")
print('  curl -X POST http://localhost:8000/ask \\')
print('       -H "Content-Type: application/json" \\')
print('       -d \'{ "question": "What is RAG?", "thread_id": "test-1" }\'')

In [ ]:
# Test FastAPI with Python requests (run AFTER starting uvicorn in a terminal)
import requests

BASE_URL = "http://localhost:8000"

def test_api():
    print("Testing FastAPI endpoints...")
    print()

    # Health check
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=5)
        if r.status_code == 200:
            data = r.json()
            print(f"✅ GET /health → status={data['status']}, kb_docs={data['kb_docs']}")
        else:
            print(f"❌ GET /health → {r.status_code}")
            return
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to FastAPI server.")
        print("   Start it first: uvicorn fastapi_app:app --reload --port 8000")
        return

    # Ask a question
    payload = {"question": "What is the sliding window approach?", "thread_id": "notebook-test"}
    r2 = requests.post(f"{BASE_URL}/ask", json=payload, timeout=30)
    data2 = r2.json()
    print(f"\n✅ POST /ask → {r2.status_code}")
    print(f"   Question:   {payload['question']}")
    print(f"   Answer:     {data2['answer'][:200]}")
    print(f"   Latency:    {data2['latency_ms']}ms")

    # Follow-up question — tests memory (same thread_id)
    payload2 = {"question": "What was my last question?", "thread_id": "notebook-test"}
    r3 = requests.post(f"{BASE_URL}/ask", json=payload2, timeout=30)
    data3 = r3.json()
    print(f"\n✅ POST /ask (follow-up) → {r3.status_code}")
    print(f"   Question:   {payload2['question']}")
    print(f"   Answer:     {data3['answer'][:200]}")

    print("\n✅ All API tests passed!")

test_api()

---
## 3. 🌊 Streaming in Streamlit

Show the agent's answer word by word — the ChatGPT typing effect. Uses `st.write_stream()` with a generator that yields chunks.

In [ ]:
streaming_app = '''
"""
streamlit_streaming.py — Streaming chatbot with word-by-word effect
Run with: streamlit run streamlit_streaming.py
"""
import streamlit as st
import uuid
import time
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

st.set_page_config(page_title="Streaming Agent", page_icon="⚡", layout="centered")
st.title("⚡ Streaming RAG Agent")
st.caption("Watch the answer appear word by word")

# ── Initialise (run once per session) ─────────────────────
@st.cache_resource
def load_models():
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    client = chromadb.Client()
    try:
        client.delete_collection("kb_stream")
    except:
        pass
    col = client.create_collection("kb_stream")

    docs = [
        "LangGraph is a library for stateful multi-actor LLM workflows using directed graphs.",
        "RAG retrieves relevant document chunks before generating an answer to ground the LLM.",
        "MemorySaver persists LangGraph state across invoke() calls using thread IDs.",
        "Self-reflection: generate answer, critique it, improve it. Max 3 iterations.",
        "Groq provides fast LLM inference at 300+ tokens/sec using custom LPU hardware.",
    ]
    col.add(documents=docs, embeddings=embedder.encode(docs).tolist(),
            ids=[f"d{i}" for i in range(len(docs))])
    return llm, embedder, col

llm, embedder, collection = load_models()

if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]

# Display history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ── Streaming generator ────────────────────────────────────
def stream_answer(question: str, messages: list):
    """Generator that yields answer chunks for st.write_stream()."""
    # Retrieve context
    q_emb   = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=2)
    context = "\\n".join(results["documents"][0])

    # Build messages
    lc_msgs = [SystemMessage(content=f"Answer using ONLY this context:\\n{context}")]
    for msg in messages[-4:]:  # last 2 turns
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    # Stream from LLM — yields chunks
    for chunk in llm.stream(lc_msgs):
        if chunk.content:
            yield chunk.content


if prompt := st.chat_input("Ask anything about Agentic AI..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        # st.write_stream() calls the generator and renders each chunk
        full_answer = st.write_stream(
            stream_answer(prompt, st.session_state.messages)
        )

    st.session_state.messages.append({"role": "assistant", "content": full_answer})
'''

with open("streamlit_streaming.py", "w", encoding="utf-8") as f:
    f.write(streaming_app)

print("✅ streamlit_streaming.py written")
print()
print("Run with: streamlit run streamlit_streaming.py")
print()
print("Key difference from streamlit_app.py:")
print("  llm.stream() instead of llm.invoke()")
print("  st.write_stream(generator) renders chunks in real-time")
print("  @st.cache_resource loads models only once per session")

---
## 4. 🔗 Connect Streamlit → FastAPI

The Streamlit frontend calls FastAPI over HTTP. The agent logic lives entirely in FastAPI — Streamlit only handles the UI.

**Setup:** Start FastAPI first (`uvicorn fastapi_app:app --reload`), then start this Streamlit app.

In [ ]:
fullstack_app = '''
"""
streamlit_fullstack.py — Streamlit frontend calling FastAPI backend

SETUP:
  Terminal 1: uvicorn fastapi_app:app --reload --port 8000
  Terminal 2: streamlit run streamlit_fullstack.py
"""
import streamlit as st
import requests
import uuid

FASTAPI_URL = "http://localhost:8000"

st.set_page_config(page_title="Full Stack Agent", page_icon="🏗️", layout="centered")
st.title("🏗️ Full Stack Agent")
st.caption("Streamlit UI → FastAPI backend → LangGraph agent")

# ── Check if backend is running ───────────────────────────
def check_backend():
    try:
        r = requests.get(f"{FASTAPI_URL}/health", timeout=3)
        return r.status_code == 200, r.json()
    except:
        return False, {}

is_up, health = check_backend()
if is_up:
    st.success(f"✅ Backend connected — {health.get('kb_docs', 0)} docs loaded")
else:
    st.error("❌ FastAPI backend not running. Start it with: uvicorn fastapi_app:app --reload")
    st.stop()  # Stop rendering if backend is down

# ── Session state ─────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("Connection")
    st.write(f"Backend: `{FASTAPI_URL}`")
    st.write(f"Thread:  `{st.session_state.thread_id}`")
    st.write(f"Model:   `{health.get('model', 'unknown')}`")

    if st.button("🗑️ New conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

# ── Display history ───────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])
        if "latency" in msg:
            st.caption(f"⏱ {msg[\'latency\']}ms")

# ── Chat input → HTTP POST to FastAPI ─────────────────────
if prompt := st.chat_input("Ask the agent..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        with st.spinner("Calling FastAPI backend..."):
            try:
                # HTTP POST to FastAPI — same as curl
                response = requests.post(
                    f"{FASTAPI_URL}/ask",
                    json={"question": prompt, "thread_id": st.session_state.thread_id},
                    timeout=30
                )
                if response.status_code == 200:
                    data = response.json()
                    st.write(data["answer"])
                    st.caption(f"⏱ {data[\'latency_ms\']}ms  |  thread: {data[\'thread_id\']}")
                    st.session_state.messages.append({
                        "role": "assistant",
                        "content": data["answer"],
                        "latency": data["latency_ms"]
                    })
                else:
                    st.error(f"Backend error {response.status_code}: {response.text[:200]}")
            except requests.exceptions.Timeout:
                st.error("Request timed out. The agent is taking too long.")
            except requests.exceptions.ConnectionError:
                st.error("Lost connection to backend. Is FastAPI still running?")
'''

with open("streamlit_fullstack.py", "w", encoding="utf-8") as f:
    f.write(fullstack_app)

print("✅ streamlit_fullstack.py written")
print()
print("To run the full stack:")
print("  Terminal 1: uvicorn fastapi_app:app --reload --port 8000")
print("  Terminal 2: streamlit run streamlit_fullstack.py")
print()
print("The Streamlit app will show a red error if FastAPI is not running.")
print("Start FastAPI first, then refresh the Streamlit page.")

---
## 5. 🔐 API Key Authentication

Add a simple API key check to FastAPI. Requests without a valid key get a 401 Unauthorized response.

In [ ]:
auth_app = '''
"""
fastapi_auth.py — FastAPI with API key authentication
Run with: uvicorn fastapi_auth:app --reload --port 8001
"""
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel
from typing import Optional
import os
from dotenv import load_dotenv
from agent import ask

load_dotenv()

app = FastAPI(title="Agentic AI API (with Auth)")

# ── API keys — in production, store in a database ─────────
# For demo: load from env or use a hardcoded set
VALID_API_KEYS = {
    "student-key-agentic2026": "student",
    "admin-key-kanthi-2026":   "admin",
}

def verify_api_key(x_api_key: Optional[str] = Header(None)) -> str:
    """Dependency: checks X-API-Key header. Returns role if valid."""
    if not x_api_key:
        raise HTTPException(
            status_code=401,
            detail="Missing X-API-Key header"
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="Invalid API key"
        )
    return VALID_API_KEYS[x_api_key]  # return the role


class AskRequest(BaseModel):
    question:  str
    thread_id: str = "default"


@app.get("/health")
def health():
    # Health check is public — no auth required
    return {"status": "ok", "auth": "enabled"}


@app.post("/ask")
def ask_question(
    request: AskRequest,
    x_api_key: Optional[str] = Header(None)
):
    """Protected endpoint — requires X-API-Key header."""
    role = verify_api_key(x_api_key)
    answer = ask(request.question, thread_id=request.thread_id)
    return {"answer": answer, "thread_id": request.thread_id, "role": role}
'''

with open("fastapi_auth.py", "w", encoding="utf-8") as f:
    f.write(auth_app)

print("✅ fastapi_auth.py written")
print()
print("Run: uvicorn fastapi_auth:app --reload --port 8001")
print()
print("Test — no key (should fail with 401):")
print('  curl -X POST http://localhost:8001/ask \\')
print('       -H "Content-Type: application/json" \\')
print('       -d \'{ "question": "What is RAG?" }\'')
print()
print("Test — valid key (should succeed):")
print('  curl -X POST http://localhost:8001/ask \\')
print('       -H "Content-Type: application/json" \\')
print('       -H "X-API-Key: student-key-agentic2026" \\')
print('       -d \'{ "question": "What is RAG?" }\'')

In [ ]:
# Test authentication (run AFTER starting fastapi_auth.py in a terminal)
import requests

AUTH_URL = "http://localhost:8001"

def test_auth():
    print("Testing API key authentication...")
    print()

    # Test 1: No API key → should get 401
    try:
        r1 = requests.post(f"{AUTH_URL}/ask",
                           json={"question": "What is RAG?"},
                           timeout=5)
        if r1.status_code == 401:
            print(f"✅ No key → 401 Unauthorized: {r1.json()['detail']}")
        else:
            print(f"⚠️  Expected 401, got {r1.status_code}")
    except requests.exceptions.ConnectionError:
        print("❌ Server not running. Start: uvicorn fastapi_auth:app --reload --port 8001")
        return

    # Test 2: Wrong API key → should get 401
    r2 = requests.post(f"{AUTH_URL}/ask",
                       json={"question": "What is RAG?"},
                       headers={"X-API-Key": "wrong-key"},
                       timeout=5)
    if r2.status_code == 401:
        print(f"✅ Wrong key → 401 Unauthorized: {r2.json()['detail']}")
    else:
        print(f"⚠️  Expected 401, got {r2.status_code}")

    # Test 3: Valid API key → should get 200
    r3 = requests.post(f"{AUTH_URL}/ask",
                       json={"question": "What is RAG?", "thread_id": "auth-test"},
                       headers={"X-API-Key": "student-key-agentic2026"},
                       timeout=30)
    if r3.status_code == 200:
        data = r3.json()
        print(f"\n✅ Valid key → 200 OK")
        print(f"   Role:   {data['role']}")
        print(f"   Answer: {data['answer'][:150]}")
    else:
        print(f"❌ Expected 200, got {r3.status_code}: {r3.text[:200]}")

test_auth()

---
## 6. 🎯 Your Turn — Challenge Exercises

### Challenge 1 (Easy): Add a topic selector to Streamlit
In `streamlit_app.py`, add a dropdown in the sidebar that lets users pick a topic (LangGraph, RAG, Memory, etc.). Pre-fill the chat input with a starter question based on the selection.

In [ ]:
# YOUR CODE HERE
# Hint: In the sidebar section of streamlit_app.py, add:
# topic = st.selectbox("Quick topic:",
#     ["--", "LangGraph", "RAG", "Memory Systems", "Self-Reflection", "Groq API"])
# starter_questions = {
#     "LangGraph": "What is LangGraph and what problems does it solve?",
#     "RAG": "Explain the four stages of a RAG pipeline.",
#     ...}
# if topic != "--":
#     st.info(f"Starter: {starter_questions[topic]}")

### Challenge 2 (Medium): Add rate limiting to FastAPI
Limit each API key to 10 requests per minute. Return 429 Too Many Requests if exceeded. Use a simple in-memory counter with timestamps.

In [ ]:
# YOUR CODE HERE
# Hint: Add to fastapi_auth.py:
# from collections import defaultdict
# import time
# request_counts = defaultdict(list)  # key → list of timestamps
#
# def check_rate_limit(api_key: str):
#     now = time.time()
#     # Keep only requests from last 60 seconds
#     request_counts[api_key] = [t for t in request_counts[api_key] if now - t < 60]
#     if len(request_counts[api_key]) >= 10:
#         raise HTTPException(status_code=429, detail="Rate limit exceeded (10/min)")
#     request_counts[api_key].append(now)
#
# Call check_rate_limit(x_api_key) at the start of the /ask endpoint

### Challenge 3 (Hard): Add conversation export to Streamlit
Add a "Download conversation" button in the sidebar that exports the full chat history as a `.txt` or `.json` file. Use `st.download_button()`.

In [ ]:
# YOUR CODE HERE
# Hint: In the sidebar of streamlit_app.py:
# import json
# if st.session_state.messages:
#     export = json.dumps(st.session_state.messages, indent=2)
#     st.download_button(
#         label="📥 Download conversation",
#         data=export,
#         file_name=f"chat_{st.session_state.thread_id}.json",
#         mime="application/json"
#     )

---
## 7. Day 12 Wrap-up

### Files you created today:
```
agent.py                  ← shared agent module (imported by all apps)
streamlit_app.py          ← full chatbot UI with session_state memory
fastapi_app.py            ← REST API with /ask and /health endpoints
streamlit_streaming.py    ← streaming chatbot (word-by-word)
streamlit_fullstack.py    ← Streamlit UI calling FastAPI backend
fastapi_auth.py           ← FastAPI with API key authentication
```

### Quick reference — how to run each:
```bash
streamlit run streamlit_app.py          # → http://localhost:8501
streamlit run streamlit_streaming.py    # → http://localhost:8501
streamlit run streamlit_fullstack.py    # → http://localhost:8501

uvicorn fastapi_app:app  --reload --port 8000  # → http://localhost:8000
uvicorn fastapi_auth:app --reload --port 8001  # → http://localhost:8001
```

### The one rule to remember:
```python
# The LangGraph agent code NEVER changes between notebook, Streamlit, and FastAPI.
# You only change the wrapper around it.
result = agent_app.invoke({"question": q}, config=config)  # always the same
```

### Tomorrow — Day 13: Capstone Project
Build your own production agent — combining everything from Days 1-12. You choose the domain, the tools, the retrieval strategy, and the deployment method.

---
### 📚 Resources:
- [Streamlit docs](https://docs.streamlit.io/)
- [Streamlit chat elements](https://docs.streamlit.io/develop/api-reference/chat)
- [FastAPI docs](https://fastapi.tiangolo.com/)
- [FastAPI tutorial](https://fastapi.tiangolo.com/tutorial/)